# Program 8: Correlation & Dispersion Monitor

**Curriculum Days: 121–135 (Portfolio Construction & Risk)**

Tracks SPY vs its top 10 holdings. Computes implied correlation (from index IV vs constituent IV) and realized correlation from rolling returns. Flags dispersion trade opportunities when implied correlation significantly exceeds realized correlation — the classic "sell index vol, buy single-stock vol" setup used by institutional desks.

**What it produces:**
- 10×10 realized correlation heatmap (30-day rolling)
- Implied vs realized correlation time series
- Dispersion opportunity signal gauge
- Sector breakdown and regime classification

## Setup

In [ ]:
!pip install yfinance numpy scipy matplotlib pandas seaborn -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import norm
import yfinance as yf
from datetime import datetime, date, timedelta

print('All imports loaded.')

## 1. Universe Configuration

**Quant Concept: Index Decomposition & Weight Attribution (Days 121–125)**

An index like SPY is a weighted portfolio. Its volatility depends on each constituent's volatility *and* the pairwise correlations between them. When correlations are low, individual stock moves cancel out (diversification), and index vol is lower than the average stock vol — creating the dispersion trade opportunity.

In [ ]:
INDEX_TICKER = "SPY"

HOLDINGS = {
    "AAPL":  {"name": "Apple",       "sector": "Technology",       "weight": 0.070},
    "MSFT":  {"name": "Microsoft",   "sector": "Technology",       "weight": 0.065},
    "NVDA":  {"name": "NVIDIA",      "sector": "Technology",       "weight": 0.058},
    "AMZN":  {"name": "Amazon",      "sector": "Consumer Disc.",   "weight": 0.040},
    "GOOGL": {"name": "Alphabet",    "sector": "Communication",    "weight": 0.037},
    "META":  {"name": "Meta",        "sector": "Communication",    "weight": 0.027},
    "TSLA":  {"name": "Tesla",       "sector": "Consumer Disc.",   "weight": 0.023},
    "BRK-B": {"name": "Berkshire",   "sector": "Financials",       "weight": 0.020},
    "JPM":   {"name": "JPMorgan",    "sector": "Financials",       "weight": 0.018},
    "V":     {"name": "Visa",        "sector": "Financials",       "weight": 0.016},
}

TICKERS = [INDEX_TICKER] + list(HOLDINGS.keys())
LOOKBACK_DAYS   = 252
ROLLING_WINDOWS = [21, 63, 126]
NEAR_TERM_EXPIRY_DAYS = 30
DISPERSION_THRESHOLD = 0.08

print(f"Tracking {len(HOLDINGS)} constituents of {INDEX_TICKER}")

## 2. Data Fetching

**Quant Concept: Implied vs Realized Measures (Days 126–130)**

We fetch two distinct data types:
- **Price history** — used to compute *realized* correlation from actual returns
- **ATM Implied Volatility** — the market's forward-looking expectation, extracted from the options chain

The gap between implied and realized is where the edge lives.

In [ ]:
def fetch_prices(tickers, lookback_days=252):
    """Download adjusted close prices for all tickers."""
    end   = date.today()
    start = end - timedelta(days=lookback_days + 60)
    print(f"  Downloading price history ({start} \u2192 {end}) for {len(tickers)} tickers...")
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True,
                      progress=False)['Close']
    if isinstance(raw, pd.Series):
        raw = raw.to_frame()
    raw.dropna(how='all', inplace=True)
    raw = raw.tail(lookback_days)
    return raw


def fetch_atm_iv(ticker, days_to_expiry=30):
    """Pull near-ATM implied volatility from options chain."""
    try:
        t = yf.Ticker(ticker)
        exps = t.options
        if not exps:
            return np.nan
        today = date.today()
        best_exp = min(exps, key=lambda e: abs(
            (datetime.strptime(e, '%Y-%m-%d').date() - today).days - days_to_expiry))
        T_days = (datetime.strptime(best_exp, '%Y-%m-%d').date() - today).days
        if T_days <= 0:
            return np.nan
        chain = t.option_chain(best_exp)
        spot  = t.history(period='1d')['Close'].iloc[-1]
        calls = chain.calls.copy()
        puts  = chain.puts.copy()
        calls['dist'] = abs(calls['strike'] - spot)
        puts['dist']  = abs(puts['strike']  - spot)
        atm_calls = calls.nsmallest(3, 'dist')
        atm_puts  = puts.nsmallest(3, 'dist')
        iv_vals = pd.concat([atm_calls['impliedVolatility'],
                             atm_puts['impliedVolatility']]).dropna()
        if iv_vals.empty:
            return np.nan
        return float(iv_vals.median())
    except Exception:
        return np.nan

print('Data fetchers ready.')

## 3. Correlation Calculations

**Quant Concept: Implied Correlation & Dispersion Trading (Days 131–135)**

**Realized correlation** is computed from historical returns using a rolling window.

**Implied correlation** is derived from the index vol formula. For a weighted index:

$$\sigma_{\text{index}}^2 = \sum_i \sum_j w_i w_j \rho_{ij} \sigma_i \sigma_j$$

Assuming uniform $\rho$ and solving:

$$\rho_{\text{implied}} = \frac{\sigma_{\text{index}}^2 - \sum_i w_i^2 \sigma_i^2}{\sum_{i \neq j} w_i w_j \sigma_i \sigma_j}$$

When $\rho_{\text{implied}} > \rho_{\text{realized}}$, the market is pricing in more correlation than exists — sell index vol, buy single-stock vol (the **dispersion trade**).

In [ ]:
def rolling_correlation_matrix(returns_df, window=21):
    recent = returns_df.tail(window).dropna()
    return recent.corr()


def rolling_avg_pairwise_corr(returns_df, window=21):
    cols = [c for c in returns_df.columns if c != INDEX_TICKER]
    sub  = returns_df[cols].dropna()
    avg_corrs = []
    dates_out = []
    for i in range(window, len(sub) + 1):
        chunk = sub.iloc[i - window:i]
        corr  = chunk.corr()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        avg_c = upper.stack().mean()
        avg_corrs.append(avg_c)
        dates_out.append(sub.index[i - 1])
    return pd.Series(avg_corrs, index=dates_out)


def implied_index_correlation(index_iv, constituent_ivs, weights):
    valid = {t: iv for t, iv in constituent_ivs.items() if not np.isnan(iv)}
    if not valid:
        return np.nan
    total_w = sum(weights[t] for t in valid)
    normalized_w = {t: weights[t] / total_w for t in valid}
    tickers = list(valid.keys())
    ivs_arr = np.array([valid[t] for t in tickers])
    w_arr   = np.array([normalized_w[t] for t in tickers])
    numerator = index_iv ** 2 - np.sum(w_arr ** 2 * ivs_arr ** 2)
    n = len(tickers)
    denom = 0.0
    for i in range(n):
        for j in range(n):
            if i != j:
                denom += w_arr[i] * w_arr[j] * ivs_arr[i] * ivs_arr[j]
    if denom == 0:
        return np.nan
    return float(np.clip(numerator / denom, -1.0, 1.0))


def dispersion_signal_score(implied_corr, realized_corr):
    if np.isnan(implied_corr) or np.isnan(realized_corr):
        return np.nan
    gap = implied_corr - realized_corr
    return float(np.clip(gap / 0.15, -1.0, 1.0))


def classify_regime(avg_realized_corr):
    if avg_realized_corr > 0.70:
        return "CRISIS / MACRO-DRIVEN", "#e74c3c"
    elif avg_realized_corr > 0.50:
        return "ELEVATED CORRELATION", "#f39c12"
    elif avg_realized_corr > 0.30:
        return "NORMAL", "#3498db"
    else:
        return "LOW CORR / STOCK-PICKING", "#2ecc71"

print('Correlation engine loaded.')

## 4. Visualization Dashboard

Four panels:
1. **Correlation Heatmap** — 21-day realized pairwise correlations
2. **Implied vs Realized Correlation** — time series with gap highlighted
3. **Dispersion Signal Gauge** — semicircle gauge showing trade attractiveness
4. **Constituent IVs** — individual stock IVs vs SPY IV

In [ ]:
def plot_dashboard(prices, returns, corr_matrices, implied_corr_val,
                   rolling_corr_series, dispersion_score,
                   constituent_ivs, index_iv, agg_realized):
    plt.style.use('dark_background')
    fig = plt.figure(figsize=(20, 16), facecolor='#0d1117')
    fig.suptitle("SPY Correlation & Dispersion Monitor",
                 fontsize=16, fontweight='bold', color='white', y=0.98)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35,
                           top=0.93, bottom=0.06, left=0.06, right=0.97)
    cmap = LinearSegmentedColormap.from_list("corr_map", ["#e74c3c", "#2a2a3e", "#2ecc71"])

    # Panel 1: Correlation Heatmap
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor('#151b27')
    non_spy_cols = [c for c in returns.columns if c != INDEX_TICKER]
    corr_sub = corr_matrices['21d'].loc[non_spy_cols, non_spy_cols]
    im = ax1.imshow(corr_sub.values, cmap=cmap, vmin=-0.5, vmax=1.0, aspect='auto')
    ax1.set_xticks(range(len(non_spy_cols)))
    ax1.set_yticks(range(len(non_spy_cols)))
    ax1.set_xticklabels(non_spy_cols, rotation=45, ha='right', color='#ccc', fontsize=8)
    ax1.set_yticklabels(non_spy_cols, color='#ccc', fontsize=8)
    ax1.set_title("21-Day Realized Correlation (Top 10)", color='white', fontsize=10, fontweight='bold', pad=8)
    for i in range(len(non_spy_cols)):
        for j in range(len(non_spy_cols)):
            val = corr_sub.values[i, j]
            ax1.text(j, i, f"{val:.2f}", ha='center', va='center',
                     color='white' if abs(val) > 0.5 else '#aaa', fontsize=7.5)
    plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04).ax.tick_params(colors='#aaa')

    # Panel 2: Implied vs Realized Time Series
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor('#151b27')
    roll_21 = rolling_corr_series.get('21d', pd.Series(dtype=float))
    roll_63 = rolling_corr_series.get('63d', pd.Series(dtype=float))
    if not roll_21.empty:
        ax2.plot(roll_21.index, roll_21.values, color='#3498db', linewidth=1.5, label='Realized (21d)')
    if not roll_63.empty:
        ax2.plot(roll_63.index, roll_63.values, color='#9b59b6', linewidth=1.5, label='Realized (63d)', alpha=0.85)
    if not np.isnan(implied_corr_val):
        ax2.axhline(implied_corr_val, color='#e74c3c', linewidth=2, linestyle='--',
                    label=f'Implied ({implied_corr_val:.2f})')
    ax2.set_ylim(-0.1, 1.05)
    ax2.set_title("Implied vs Realized Correlation", color='white', fontsize=10, fontweight='bold', pad=8)
    ax2.legend(fontsize=8, facecolor='#1a1f2e', labelcolor='white', framealpha=0.8)
    ax2.tick_params(colors='#ccc', labelsize=8)
    for spine in ax2.spines.values():
        spine.set_edgecolor('#333')

    # Panel 3: Dispersion Gauge
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_facecolor('#151b27')
    theta = np.linspace(0, np.pi, 200)
    zone_boundaries = [0, 0.3, 0.6, 1.0]
    zone_colors = ['#e74c3c', '#f39c12', '#2ecc71']
    for k in range(3):
        t1 = np.pi * (1 - zone_boundaries[k+1])
        t2 = np.pi * (1 - zone_boundaries[k])
        tz = np.linspace(t1, t2, 50)
        ax3.fill_between(np.cos(tz), np.sin(tz), np.cos(tz)*0.7, np.sin(tz)*0.7,
                         color=zone_colors[k], alpha=0.4)
    ax3.plot(np.cos(theta), np.sin(theta), color='#555', linewidth=2)
    ax3.plot(np.cos(theta)*0.7, np.sin(theta)*0.7, color='#555', linewidth=2)
    sc = float(np.clip(dispersion_score if not np.isnan(dispersion_score) else 0, 0, 1))
    na = np.pi * (1 - sc)
    ax3.annotate("", xy=(np.cos(na)*0.85, np.sin(na)*0.85), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='white', lw=2.5))
    ax3.set_xlim(-1.2, 1.2)
    ax3.set_ylim(-0.15, 1.2)
    ax3.axis('off')
    ax3.set_title("Dispersion Opportunity Signal", color='white', fontsize=10, fontweight='bold', pad=8)
    label = "STRONG" if sc > 0.67 else "MODERATE" if sc > 0.33 else "WEAK"
    lcolor = "#2ecc71" if sc > 0.67 else "#f39c12" if sc > 0.33 else "#e74c3c"
    ax3.text(0, -0.1, f"{label} SIGNAL", ha='center', color=lcolor, fontsize=12, fontweight='bold')
    ax3.text(0, -0.3, f"Impl={implied_corr_val:.2f}  Real={agg_realized:.2f}  Gap={implied_corr_val-agg_realized:.2f}",
             ha='center', color='#aaa', fontsize=9)

    # Panel 4: Constituent IVs
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_facecolor('#151b27')
    sector_colors = {"Technology": "#3498db", "Consumer Disc.": "#e67e22",
                     "Communication": "#9b59b6", "Financials": "#2ecc71"}
    tickers_sorted = sorted(constituent_ivs.keys(),
                            key=lambda t: constituent_ivs.get(t, 0) or 0, reverse=True)
    valid_t = [t for t in tickers_sorted if not np.isnan(constituent_ivs.get(t, np.nan))]
    valid_iv = [constituent_ivs[t] for t in valid_t]
    valid_c = [sector_colors.get(HOLDINGS[t]['sector'], '#aaa') for t in valid_t]
    bars = ax4.barh(valid_t, [v*100 for v in valid_iv], color=valid_c, edgecolor='#333', height=0.6)
    if not np.isnan(index_iv):
        ax4.axvline(index_iv*100, color='white', linewidth=2, linestyle='--',
                    label=f'SPY IV ({index_iv*100:.1f}%)')
        ax4.legend(fontsize=8, facecolor='#1a1f2e', labelcolor='white')
    ax4.set_title("Constituent IV vs SPY IV (30-day ATM)", color='white', fontsize=10, fontweight='bold', pad=8)
    ax4.tick_params(colors='#ccc', labelsize=9)
    for spine in ax4.spines.values():
        spine.set_edgecolor('#333')

    plt.tight_layout()
    plt.show()

print('Dashboard renderer loaded.')

## 5. Run the Correlation Monitor

This fetches all data, computes correlations, extracts IVs, and renders the full dashboard.

In [ ]:
print("=" * 65)
print("  PROGRAM 8: Correlation & Dispersion Monitor")
print("=" * 65)

print("\n[1/5] Fetching price history...")
prices = fetch_prices(TICKERS, lookback_days=LOOKBACK_DAYS)
returns = prices.pct_change().dropna()
print(f"  Got {len(prices)} trading days for {len(prices.columns)} tickers.")

print("\n[2/5] Computing rolling correlation matrices...")
corr_matrices = {}
rolling_corr_series = {}
for w in ROLLING_WINDOWS:
    label = f"{w}d"
    corr_matrices[label] = rolling_correlation_matrix(returns, window=w)
    rolling_corr_series[label] = rolling_avg_pairwise_corr(returns, window=w)
    avg = rolling_corr_series[label].iloc[-1] if len(rolling_corr_series[label]) > 0 else np.nan
    print(f"  {label} avg pairwise corr: {avg:.3f}")

agg_realized_21 = float(rolling_corr_series['21d'].iloc[-1]) \
    if len(rolling_corr_series['21d']) > 0 else np.nan

print("\n[3/5] Fetching implied volatilities (~30 seconds)...")
constituent_ivs = {}
for ticker in HOLDINGS.keys():
    iv = fetch_atm_iv(ticker, days_to_expiry=NEAR_TERM_EXPIRY_DAYS)
    constituent_ivs[ticker] = iv
    status = f"{iv*100:.1f}%" if not np.isnan(iv) else "N/A"
    print(f"  {ticker:6s}: ATM IV = {status}")

index_iv = fetch_atm_iv(INDEX_TICKER, days_to_expiry=NEAR_TERM_EXPIRY_DAYS)
print(f"  {INDEX_TICKER:6s}: ATM IV = {index_iv*100:.1f}%" if not np.isnan(index_iv) else f"  {INDEX_TICKER}: IV unavailable")

print("\n[4/5] Computing implied correlation...")
weights = {t: HOLDINGS[t]['weight'] for t in HOLDINGS}
implied_corr = implied_index_correlation(index_iv, constituent_ivs, weights)
if not np.isnan(implied_corr):
    print(f"  Implied Correlation: {implied_corr:.3f}")
else:
    print("  Implied Correlation: N/A")
    implied_corr = agg_realized_21

gap = implied_corr - agg_realized_21 if not np.isnan(agg_realized_21) else np.nan
d_score = dispersion_signal_score(implied_corr, agg_realized_21)

print("\n[5/5] Classifying market regime...")
regime_label, _ = classify_regime(agg_realized_21)
print(f"  21d Realized Corr: {agg_realized_21:.3f}")
print(f"  Implied Corr:      {implied_corr:.3f}")
print(f"  Dispersion Gap:    {gap:.3f}" if not np.isnan(gap) else "  Gap: N/A")
print(f"  Market Regime:     {regime_label}")

print("\nRendering dashboard...")
plot_dashboard(prices, returns, corr_matrices, implied_corr,
               rolling_corr_series, d_score, constituent_ivs, index_iv, agg_realized_21)
print("Done.")

## Exercise 1: Change the Rolling Window

Modify `ROLLING_WINDOWS` to use `[10, 30, 60]` instead. Shorter windows are more responsive but noisier. Observe how the 10-day correlation heatmap differs from the 63-day one. Which window best captures the current regime?

In [ ]:
# YOUR CODE HERE: Change ROLLING_WINDOWS and re-run


## Exercise 2: Swap the Universe

Replace the SPY top 10 with QQQ top 10 (tech-heavy). How does the average pairwise correlation change when the universe is more sector-concentrated? Tech stocks tend to be more correlated — does this reduce the dispersion opportunity?

In [ ]:
# YOUR CODE HERE: Change HOLDINGS to QQQ top holdings and re-run


## Exercise 3: Adjust the Dispersion Threshold

The current `DISPERSION_THRESHOLD = 0.08` means we flag a signal when implied corr exceeds realized by 8 points. Try lowering to 0.04 (more sensitive) or raising to 0.12 (fewer but stronger signals). What's the trade-off between signal frequency and quality?

In [ ]:
# YOUR CODE HERE: Adjust DISPERSION_THRESHOLD


## Key Takeaways

1. **Implied correlation > realized correlation = dispersion opportunity** — The market is pricing in more co-movement than actually exists. Sell index vol (overpriced), buy stock vol (underpriced).

2. **Regime matters** — In crisis mode (corr > 0.70), everything moves together and dispersion trades lose. The best dispersion environments are normal-to-low correlation regimes.

3. **Index vol is structurally cheap relative to stock vol** — This is because diversification mechanically reduces index vol. The vol risk premium is larger at the index level.

4. **The dispersion trade is the institutional version of selling vol** — Instead of naked short vol, you're hedged by being long single-stock vol. It's a *relative value* bet on correlation.

5. **Poker parallel** — Implied correlation is like the pot odds the table is offering. Realized correlation is the actual hand distribution. When the gap is wide, the table is mispricing the game.